# Dandy Senior Data Engineer Interview — Python (30 min)

**Candidate:** Rahul Raj Singh

---
## Problem 1: Dental Lab Order Schema Normalizer

**Difficulty:** Easy | **Focus:** Data Cleaning & String Manipulation

**Scenario:** Dandy receives order metadata from various clinics in inconsistent formats (CamelCase, snake_case, or kebab-case).

**Task:** Normalize all keys to snake_case and ensure all price values are floats. If a price is missing or invalid, default it to 0.0.

**Input:** `[{"OrderId": 101, "patient-name": "Alice", "Total_Price": "150.50"}]`

**Output:** `[{"order_id": 101, "patient_name": "Alice", "total_price": 150.50}]`

**Edge cases:** None, empty strings, non-numeric strings for prices

In [ ]:
import pandas as pd
import re

def normalize_orders(orders: list[dict]) -> list[dict]:
    """Normalize order keys to snake_case and ensure price values are floats."""
    
    normalized_list = []
    
    for input_dict in orders:
        if not input_dict:
            normalized_list.append({})
            continue
            
        output_dict = {}
        for key, value in input_dict.items():
            # Step 1: Convert CamelCase to snake_case (e.g., OrderId -> order_id)
            # Insert underscore before each uppercase letter, then strip leading underscore
            snake_key = re.sub(r'([A-Z])', r'_\1', key).lstrip('_')
            
            # Step 2: Replace hyphens and spaces with underscores (e.g., patient-name -> patient_name)
            snake_key = re.sub(r'[-\s]', '_', snake_key)
            
            # Step 3: Collapse underscores, strip leading/trailing, lowercase
            snake_key = re.sub(r'_+', '_', snake_key).strip('_').lower()

            # Only convert price-related values to floats; keep other values as-is
            if 'price' in snake_key.lower():
                processed_value = 0.0
                if isinstance(value, (int, float)):
                    processed_value = float(value)
                elif isinstance(value, str):
                    cleaned_value = value.strip()
                    if re.fullmatch(r"[-+]?\d*\.?\d+", cleaned_value):
                        try:
                            processed_value = float(cleaned_value)
                        except ValueError:
                            pass
                output_dict[snake_key] = processed_value
            else:
                output_dict[snake_key] = value
        normalized_list.append(output_dict)
    return normalized_list

In [ ]:
# Test cases
# normalize_orders([{"OrderId": 101, "patient-name": "Alice", "Total_Price": "150.50"}])
# normalize_orders([{"Total_Price": None}])
# normalize_orders([{"Total_Price": "invalid"}])

---
## Problem 2: Impression-to-Delivery Bottleneck Detector

**Difficulty:** Medium | **Focus:** State Management & Datetime Math

**Scenario:** A dental "Case" moves through stages: Scanning → Design → Milling → Shipping.

**Task:** Identify "Zombies"—cases that started Scanning but haven't had a status update in over 24 hours relative to a provided `current_time`.

**Input:** A list of tuples `(case_id, status, timestamp_iso)` and a `current_time` string. Events are not sorted.

**Output:** List of case_ids that are zombies.

In [ ]:
import pandas as pd

def find_zombie_cases(events: list[tuple[str, str, str]], current_time: str) -> list[str]:
    """Return case_ids whose LATEST status is Scanning and last update was 24+ hours ago."""

    case_df = pd.DataFrame(events, columns=['case_id', 'status', 'timestamp_iso'])
    case_df['timestamp_iso'] = pd.to_datetime(case_df['timestamp_iso'], utc=True)

    # Get the latest event per case (events are unsorted)
    latest_per_case = (
        case_df.sort_values('timestamp_iso')
        .groupby('case_id', as_index=False)
        .last()
    )

    # Zombie = latest status is Scanning AND last update > 24h ago
    current_time_dt = pd.to_datetime(current_time, utc=True)
    cutoff = current_time_dt - pd.Timedelta(days=1)

    zombie_mask = (
        (latest_per_case['status'] == 'Scanning') &
        (latest_per_case['timestamp_iso'] < cutoff)
    )
    return latest_per_case.loc[zombie_mask, 'case_id'].tolist()

**Alternative: Pure Python (no Pandas)**

In [ ]:
from datetime import datetime, timedelta

def find_zombie_cases_pure(events: list[tuple[str, str, str]], current_time: str) -> list[str]:
    """Return case_ids whose LATEST status is Scanning and last update was 24+ hours ago."""

    # Build latest event per case (events are unsorted)
    # Type hint: dict[str, tuple[str, datetime]]
    #   - Keys (str): case_id
    #   - Values (tuple[str, datetime]): (status, timestamp) of that case's latest event
    # We overwrite when we see a newer timestamp, so we end up with only the latest per case
    latest: dict[str, tuple[str, datetime]] = {}
    for case_id, status, ts_str in events:
        ts = datetime.fromisoformat(ts_str.replace("Z", "+00:00"))
        if case_id not in latest or ts > latest[case_id][1]:
            latest[case_id] = (status, ts)

    current_dt = datetime.fromisoformat(current_time.replace("Z", "+00:00"))
    cutoff = current_dt - timedelta(days=1)

    return [
        case_id
        for case_id, (status, ts) in latest.items()
        if status == "Scanning" and ts < cutoff
    ]

In [ ]:
# Test cases
events = [
    # C001: Active - moved to Design within 24h (latest: 2025-03-15 12:00)
    ("C001", "Scanning", "2025-03-15T10:00:00Z"),
    ("C001", "Design", "2025-03-15T12:00:00Z"),
    # C002: Zombie - stuck in Scanning since 2025-03-14 08:00 (>24h ago)
    ("C002", "Scanning", "2025-03-14T08:00:00Z"),
    # C003: Zombie - Scanning only, last update 2025-03-13 14:00
    ("C003", "Scanning", "2025-03-13T14:00:00Z"),
    # C004: Active - progressed to Milling (latest: 2025-03-16 08:00)
    ("C004", "Scanning", "2025-03-15T09:00:00Z"),
    ("C004", "Design", "2025-03-15T16:00:00Z"),
    ("C004", "Milling", "2025-03-16T08:00:00Z"),
    # C005: Zombie - events out of order; latest is Scanning at 2025-03-12
    ("C005", "Design", "2025-03-11T10:00:00Z"),
    ("C005", "Scanning", "2025-03-12T06:00:00Z"),
]
current_time = "2025-03-16T09:00:00Z"
print("Pandas:", find_zombie_cases(events, current_time))
print("Pure Python:", find_zombie_cases_pure(events, current_time))  # Expected: ["C002", "C003", "C005"]